# Demo: PyTorch Lightning vs fastai on the same task

Same small CNN, same MNIST data loaders, 2 epochs. Part A isolates the training layer. Part B shows fastai's pretrained transfer learning.

**Status: executed top to bottom on Nov 27, 2026, on a CPU-only machine (2 cores, 8 GB RAM). Outputs are saved below.**

Install (GPU optional): `pip install torch torchvision lightning fastai`

In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST("data", train=True, download=True, transform=tfm)
valid_ds = datasets.MNIST("data", train=False, download=True, transform=tfm)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
valid_dl = DataLoader(valid_ds, batch_size=256, num_workers=2)

def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 7 * 7, 10),
    )

EPOCHS = 2
results = {}

device: cpu


## Part A1: PyTorch Lightning
You own the model and the step logic. The Trainer owns the loop.

In [2]:
import lightning as L

class LitCNN(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = make_cnn()

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = F.cross_entropy(self.net(x), y)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self.net(x)
        self.log("val_acc", (logits.argmax(1) == y).float().mean(), prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

lit_model = LitCNN()
trainer = L.Trainer(max_epochs=EPOCHS, accelerator="auto", devices=1,
                    logger=False, enable_checkpointing=False)
t0 = time.perf_counter()
trainer.fit(lit_model, train_dataloaders=train_dl, val_dataloaders=valid_dl)
lit_time = time.perf_counter() - t0
lit_acc = trainer.callback_metrics["val_acc"].item()
results["Lightning"] = (lit_time, lit_acc)
print(f"Lightning: {lit_time:.1f}s, val acc {lit_acc:.4f}")

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net  │ Sequential │ 20.5 K │ train │     0 │
└───┴──────┴────────────┴────────┴───────┴───────┘

Trainable params: 20.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 20.5 K                                                                                               
Total estimated model params size (MB): 0.082                                                                      
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/tmp/nbv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/tmp/nbv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` 
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

`Trainer.fit` stopped: `max_epochs=2` reached.


Lightning: 34.1s, val acc 0.9845


## Part A2: fastai, same model and loaders
fastai wraps plain PyTorch modules and DataLoaders in a `Learner` (the migration path shown in its docs).

In [3]:
from fastai.vision.all import DataLoaders, Learner, accuracy

dls = DataLoaders(train_dl, valid_dl, device=device)
learn = Learner(dls, make_cnn(), loss_func=F.cross_entropy, metrics=accuracy)
t0 = time.perf_counter()
learn.fit_one_cycle(EPOCHS, 1e-3)
fa_time = time.perf_counter() - t0
fa_acc = float(learn.validate()[1])
results["fastai"] = (fa_time, fa_acc)
print(f"fastai: {fa_time:.1f}s, val acc {fa_acc:.4f}")

/tmp/nbv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


epoch,train_loss,valid_loss,accuracy,time
0,0.111274,0.086342,0.974200,00:16
1,0.069364,0.056661,0.981500,00:17


fastai: 34.4s, val acc 0.9815


## Part A summary
Note the two runs differ in schedule (Adam fixed lr vs one-cycle), so treat accuracy differences as a learning point, not a verdict on the libraries.

In [4]:
for name, (t, a) in results.items():
    print(f"{name:10s} time={t:6.1f}s  accuracy={a:.4f}")

Lightning  time=  34.1s  accuracy=0.9845
fastai     time=  34.4s  accuracy=0.9815


## Part B: fastai transfer learning (from the official quick start)
Pretrained ResNet34, fine-tuned. Needs a download of the Pets dataset and weights.

I ran this on a CPU-only machine with 8 GB of RAM, so the batch size is 32 and the loader runs without worker processes. The default settings ran out of memory there. On a Colab GPU the defaults are fine.

In [5]:
from fastai.vision.all import *

path = untar_data(URLs.PETS) / "images"
def is_cat(x): return x[0].isupper()
pets = ImageDataLoaders.from_name_func(
    path, get_image_files(path), valid_pct=0.2, seed=42,
    label_func=is_cat, item_tfms=Resize(224),
    bs=32, num_workers=0)  # smaller batch, no worker processes: fits in 8 GB RAM on CPU
learn2 = vision_learner(pets, resnet34, metrics=error_rate)
learn2.fine_tune(1)

epoch,train_loss,valid_loss,error_rate,time
0,0.163308,0.032198,0.009472,10:35


epoch,train_loss,valid_loss,error_rate,time
0,0.057412,0.009650,0.004736,13:54


## Results from my run (CPU only)
- Training time, 2 epochs on MNIST: Lightning 34.1 s, fastai 34.4 s
- Validation accuracy: Lightning 0.9845, fastai 0.9815
- Lines of code I wrote for training (not counting the shared model and loaders): Lightning 25, fastai 9
- Part B, pretrained ResNet34 fine-tuned for 1 epoch on cats vs dogs: error rate 0.0047 (about 24 minutes on CPU)
- What the numbers say: on the same model and data the two libraries land in the same place. The small accuracy gap comes from different schedules (fixed-rate Adam vs one-cycle), so it is not a verdict on either library.